# NB11 — Active learning classifier cho EVALUATION3 overlap

Notebook này **không chạy lại toàn bộ ảnh**. Nó dùng pair evidence + preview đã được NB10D tạo ra để thử đúng ý tưởng:

`human labels nhỏ -> train LR / RBF-SVM / Random Forest -> hỏi các pair model không chắc -> human label -> train lại -> ... -> final test`

Điểm rất quan trọng về methodology:
- các batch model hỏi thêm **không gọi là test set**; chúng là `query/manual-review batches` và sau khi human label thì được thêm vào TRAIN;
- một `validation set` cố định được dùng để chọn model + hai threshold;
- một `final test` cố định được giữ kín khỏi train/validation và chỉ chạy **một lần cuối** sau khi model + thresholds đã freeze;
- split được giữ group-disjoint theo `e3_outfit_id` để cùng outfit không chui sang train và validation/test.

Classifier target: `DUPLICATE=1`, `NON_DUPLICATE=0`. Thay vì ép một threshold duy nhất, NB11 chọn **hai threshold** trên validation:
- xác suất rất thấp -> AUTO NON_DUPLICATE;
- xác suất rất cao -> AUTO DUPLICATE;
- ở giữa -> MANUAL_REVIEW.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
BRANCH = 'feat/evaluation3-active-learning-nb11'
REPO_ROOT = Path('/content/opisoverated-e3-nb11')

def run_git(*args, cwd=None):
    return subprocess.run(['git', '-c', 'http.version=HTTP/1.1', *args], cwd=cwd, check=True, text=True)

if not (REPO_ROOT / '.git').is_dir():
    run_git('clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT))
else:
    run_git('fetch', 'origin', BRANCH, cwd=REPO_ROOT)
    run_git('switch', BRANCH, cwd=REPO_ROOT)
    run_git('pull', '--ff-only', 'origin', BRANCH, cwd=REPO_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-evaluation.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.evaluation3_active_learning import (
    DUPLICATE, NON_DUPLICATE,
    apply_triage, binary_metrics, choose_triage_thresholds,
    diverse_batch, fit_compare_models, merge_review_labels,
    prepare_feature_frame, read_review_file, uncertainty_batch,
    write_review_workbook,
)
from IPython.display import display
print('BRANCH:', BRANCH)


## 1. Paths + cấu hình pilot

NB11 dùng `evaluation3_manual_review_KEY.csv` của NB10D vì file đó có cả feature và `preview_file`. Nghĩa là experiment đầu tiên học **trên vùng hard/manual của NB10D**, đúng chỗ ta đang muốn giảm công review.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
ECC_OUTPUT_DIR = DRIVE_ROOT / 'evaluation3_overlap_ecc_verifier_v0' / 'ecc_consensus_trial'
KEY_CSV = ECC_OUTPUT_DIR / 'evaluation3_manual_review_KEY.csv'
PREVIEW_DIR = ECC_OUTPUT_DIR / 'manual_review_previews'

WORK_DIR = DRIVE_ROOT / 'evaluation3_active_learning_nb11'
WORK_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = WORK_DIR / 'nb11_state.json'

RANDOM_STATE = 42
SEED_SIZE = 80
VALIDATION_SIZE = 50
FINAL_TEST_SIZE = 50
ROUND_BATCH_SIZE = 40

# Validation-only triage goals. Có thể nới sau khi có đủ labels.
TARGET_AUTO_DUP_PRECISION = 0.98
TARGET_AUTO_NON_NPV = 0.995
MIN_AUTO_VALIDATION_EXAMPLES = 5

print('KEY_CSV:', KEY_CSV)
print('WORK_DIR:', WORK_DIR)
if not KEY_CSV.is_file():
    raise FileNotFoundError(
        'Chưa có NB10D manual KEY. Cần chạy NB10D prepare ít nhất một lần trước. ' + str(KEY_CSV)
    )


## 2. Khởi tạo split cố định + 3 workbook đầu

Lần đầu chạy cell này, notebook sẽ giữ riêng outfit groups cho `validation` và `final_test`, rồi chọn một seed đa dạng từ phần còn lại. Sau đó state được lưu vào Drive nên rerun không random split lại.


In [ ]:
key = pd.read_csv(KEY_CSV)
key['pair_id'] = key['pair_id'].astype(str).str.strip()
key['e3_outfit_id'] = key['e3_outfit_id'].astype(str).str.strip()
print('Manual-pair universe:', len(key), 'pairs /', key['e3_outfit_id'].nunique(), 'outfits')

if not STATE_FILE.is_file():
    final_batch = diverse_batch(
        key, batch_size=min(FINAL_TEST_SIZE, len(key)),
        group_column='e3_outfit_id', random_state=RANDOM_STATE + 1,
    )
    final_outfits = set(final_batch['e3_outfit_id'].astype(str))
    remaining = key[~key['e3_outfit_id'].isin(final_outfits)].copy()

    validation_batch = diverse_batch(
        remaining, batch_size=min(VALIDATION_SIZE, len(remaining)),
        group_column='e3_outfit_id', random_state=RANDOM_STATE + 2,
    )
    validation_outfits = set(validation_batch['e3_outfit_id'].astype(str))
    train_pool = remaining[~remaining['e3_outfit_id'].isin(validation_outfits)].copy()

    seed_batch = diverse_batch(
        train_pool, batch_size=min(SEED_SIZE, len(train_pool)),
        group_column='e3_outfit_id', random_state=RANDOM_STATE + 3,
    )

    state = {
        'random_state': RANDOM_STATE,
        'final_test_pair_ids': final_batch['pair_id'].tolist(),
        'final_test_outfits': sorted(final_outfits),
        'validation_pair_ids': validation_batch['pair_id'].tolist(),
        'validation_outfits': sorted(validation_outfits),
        'seed_pair_ids': seed_batch['pair_id'].tolist(),
    }
    STATE_FILE.write_text(json.dumps(state, indent=2), encoding='utf-8')

    write_review_workbook(
        final_batch, WORK_DIR / 'FINAL_TEST_DO_NOT_USE_YET.xlsx',
        title='NB11 FINAL TEST - do not use for training/model selection',
    )
    write_review_workbook(
        validation_batch, WORK_DIR / 'VALIDATION_FIXED.xlsx',
        title='NB11 fixed validation set',
    )
    write_review_workbook(
        seed_batch, WORK_DIR / 'round_00_seed.xlsx',
        title='NB11 round 00 diverse seed',
    )
    print('Initialized NB11 state + review workbooks.')
else:
    state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
    print('Using existing fixed state:', STATE_FILE)

print(json.dumps({
    'seed_pairs': len(state['seed_pair_ids']),
    'validation_pairs': len(state['validation_pair_ids']),
    'final_test_pairs': len(state['final_test_pair_ids']),
    'validation_outfits': len(state['validation_outfits']),
    'final_test_outfits': len(state['final_test_outfits']),
}, indent=2))
print('\nHuman task đầu tiên: label round_00_seed.xlsx + VALIDATION_FIXED.xlsx.')
print('FINAL_TEST_DO_NOT_USE_YET.xlsx có thể label trước nếu muốn, nhưng notebook sẽ không đọc nó cho tới cell final test.')


## 3. Import labels đã review

Mỗi round chỉ cần mở workbook tương ứng trong Drive, nhìn `preview_file`, điền `DUPLICATE` hoặc `NON_DUPLICATE`, save, rồi rerun từ cell này.


In [ ]:
state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
validation_outfits = set(map(str, state['validation_outfits']))
final_test_outfits = set(map(str, state['final_test_outfits']))
reserved_outfits = validation_outfits | final_test_outfits

train_pool_all = key[~key['e3_outfit_id'].isin(reserved_outfits)].copy()
round_files = sorted(WORK_DIR.glob('round_*.xlsx'))
train_with_labels = merge_review_labels(train_pool_all, round_files)
train_labeled = train_with_labels[train_with_labels['target'].notna()].copy()
train_unlabeled = train_with_labels[train_with_labels['target'].isna()].copy()

validation_review = WORK_DIR / 'VALIDATION_FIXED.xlsx'
validation_key = key[key['pair_id'].isin(state['validation_pair_ids'])].copy()
validation_with_labels = merge_review_labels(validation_key, [validation_review])
validation_labeled = validation_with_labels[validation_with_labels['target'].notna()].copy()

print('Training labels:', len(train_labeled), '/', len(train_pool_all))
print(train_labeled['human_label'].value_counts(dropna=False).to_dict())
print('Validation labels:', len(validation_labeled), '/', len(validation_key))
print(validation_labeled['human_label'].value_counts(dropna=False).to_dict())
print('Remaining unlabeled training pool:', len(train_unlabeled))


## 4. Train Logistic Regression / RBF-SVM / Random Forest

Validation phải được label đủ trước khi chọn model/threshold. Mỗi round notebook train lại cả 3 model trên **toàn bộ training labels đã tích lũy**, rồi rank trên cùng fixed validation.


In [ ]:
fitted_models = {}
model_report = None
best_model = None
best_model_name = None
triage_thresholds = None

ready = (
    len(train_labeled) >= 20
    and train_labeled['target'].nunique() == 2
    and len(validation_labeled) == len(validation_key)
    and validation_labeled['target'].nunique() == 2
)

if not ready:
    print('CHƯA TRAIN ĐƯỢC.')
    print('- cần >=20 training labels và có cả DUP/NON')
    print('- cần label đủ VALIDATION_FIXED.xlsx và validation có cả 2 class')
else:
    fitted_models, model_report = fit_compare_models(
        train_labeled, validation_labeled, random_state=RANDOM_STATE
    )
    display(model_report)
    best_model_name = str(model_report.iloc[0]['model'])
    best_model = fitted_models[best_model_name]

    val_x = prepare_feature_frame(validation_labeled)
    classes = list(best_model.classes_)
    pos_idx = classes.index(1)
    val_prob = best_model.predict_proba(val_x)[:, pos_idx]
    triage_thresholds = choose_triage_thresholds(
        validation_labeled['target'].astype(int).to_numpy(),
        val_prob,
        target_auto_duplicate_precision=TARGET_AUTO_DUP_PRECISION,
        target_auto_non_npv=TARGET_AUTO_NON_NPV,
        minimum_auto_examples=MIN_AUTO_VALIDATION_EXAMPLES,
    )
    print('BEST MODEL:', best_model_name)
    print('TRIAGE THRESHOLDS:', triage_thresholds)
    print('Interpretation:')
    print(' p <=', triage_thresholds.auto_non_max_probability, '=> AUTO NON')
    print(' p >=', triage_thresholds.auto_duplicate_min_probability, '=> AUTO DUP')
    print(' middle => MANUAL')


## 5. Active-learning round tiếp theo

Model lấy những pair có `P(DUP)` gần 0.5 nhất trong pool chưa label. Đây là các pair model **không chắc nhất**, nên mỗi manual label có giá trị học cao hơn việc review tuần tự hàng nghìn pair.


In [ ]:
CREATE_NEXT_ROUND = True

if not ready or not CREATE_NEXT_ROUND:
    print('SKIPPED next-round creation')
elif train_unlabeled.empty:
    print('Không còn pair unlabeled trong active-learning pool.')
else:
    existing_round_numbers = []
    for path in WORK_DIR.glob('round_*.xlsx'):
        try:
            existing_round_numbers.append(int(path.stem.split('_')[1]))
        except Exception:
            pass
    next_round = max(existing_round_numbers, default=0) + 1
    destination = WORK_DIR / f'round_{next_round:02d}_query.xlsx'

    if destination.is_file():
        print('Round workbook đã tồn tại, không overwrite:', destination)
    else:
        query = uncertainty_batch(
            best_model, train_unlabeled,
            batch_size=min(ROUND_BATCH_SIZE, len(train_unlabeled)),
            group_column='e3_outfit_id',
        )
        write_review_workbook(
            query, destination,
            title=f'NB11 active-learning round {next_round:02d}',
        )
        print('Created:', destination)
        print('Label workbook này rồi rerun cells 3 -> 5. Model sẽ học thêm từ chính labels mới.')
        display(query[['pair_id','preview_file','model_probability_duplicate','rgb_ssim','edge_ssim','mean_lab_delta','patch_mae_max']].head(20))


## 6. FINAL TEST — chỉ bật sau khi đã freeze quyết định

Đừng chạy cell này mỗi round. Chỉ khi bạn đã quyết định: model nào, dừng active learning ở round nào, target precision/NPV và thresholds validation nào. Khi đó set `RUN_FINAL_TEST=True`. Final test **không được đưa ngược vào train** nếu còn muốn gọi kết quả đó là final test.


In [ ]:
RUN_FINAL_TEST = False

if not RUN_FINAL_TEST:
    print('Final test đang khóa. Đây là trạng thái đúng trong lúc active learning.')
elif not ready:
    print('Model/validation chưa ready.')
else:
    final_review = WORK_DIR / 'FINAL_TEST_DO_NOT_USE_YET.xlsx'
    final_key = key[key['pair_id'].isin(state['final_test_pair_ids'])].copy()
    final_with_labels = merge_review_labels(final_key, [final_review])
    final_labeled = final_with_labels[final_with_labels['target'].notna()].copy()
    if len(final_labeled) != len(final_key):
        print('Chưa label đủ FINAL TEST:', len(final_labeled), '/', len(final_key))
    elif final_labeled['target'].nunique() < 2:
        print('Final test chỉ có 1 class; sample này không đủ để đánh giá classifier.')
    else:
        final_x = prepare_feature_frame(final_labeled)
        pos_idx = list(best_model.classes_).index(1)
        final_prob = best_model.predict_proba(final_x)[:, pos_idx]
        print('FINAL binary metrics:')
        print(json.dumps(binary_metrics(final_labeled['target'].astype(int), final_prob), indent=2))

        triage = apply_triage(final_prob, triage_thresholds)
        y = final_labeled['target'].astype(int).to_numpy()
        auto_dup = triage == DUPLICATE
        auto_non = triage == NON_DUPLICATE
        manual = ~(auto_dup | auto_non)
        report = {
            'best_model': best_model_name,
            'n': len(y),
            'auto_dup_count': int(auto_dup.sum()),
            'auto_dup_precision': float(y[auto_dup].mean()) if auto_dup.any() else None,
            'auto_non_count': int(auto_non.sum()),
            'auto_non_npv': float((1-y[auto_non]).mean()) if auto_non.any() else None,
            'manual_count': int(manual.sum()),
            'manual_fraction': float(manual.mean()),
            'thresholds': triage_thresholds.__dict__,
        }
        print('FINAL triage report:')
        print(json.dumps(report, indent=2))
